## 필요 라이브러리 호출

In [2]:
!pip install wordcloud

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re #Regular Expression 특수문자, 반복 패턴 처리 용이
from collections import Counter

#시각화
import matplotlib.pyplot as plt
import seaborn as sns

from wordcloud import WordCloud
font_path = 'C:/Windows/Fonts/malgun.ttf' 

wc = WordCloud(
    font_path=font_path,
    background_color='white',
    width=800,
    height=600
)

import plotly.graph_objects as go  # Sankey Diagram

from konlpy.tag import Okt
'''
okt = Okt()
nouns = okt.nouns("맛있는 호박고구마 5kg 특가")
print(nouns) # 출력: ['호박고구마', '특가']
'''

plt.rcParams['font.family'] = ['Malgun Gothic', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

%matplotlib inline

## 변수 설명

product_name: 1차 정제 제품명

category_large: 대분류 카테고리

category_medium: 중분류 카테고리

item_type: 제품의 카테고리 태그

grocery_item_name: 식재료 이름(타겟)

brand_name: 제품의 브랜드 명

## 데이터 전처리

In [2]:
grocery_df = pd.read_csv('C:/Users/LG/OneDrive/Desktop/REF_Classification_For_Ingredient_Recognition/ML_dataset_smapled/refined_grocery_dataset_sampled/ML_grocery_data_sampled(CSV).csv')

print("Grocery data shape:", grocery_df.shape)

display(grocery_df.head(5))

grocery_df.info()

Grocery data shape: (1143, 6)


,product_name,category_large,category_medium,item_type,grocery_item_name,brand_name
0,[마이너피겨스] 유기농 오트음료,음료,주스,BEVERAGE,오트음료,마이너피겨스
1,[마이셰프] 찹스테이크,간편식,밀키트,RTC_MEAL,밀키트,마이셰프
2,[건강한우리집비옴] 생 아몬드 분말,조미료,가루분말,SAUCE_SEASONING,아몬드 분말,건강한우리집비옴
3,[김구원선생] 매일 마시는 국산 콩물,채소/곡물류,콩/두부,SIMPLE_PROCESSED,콩물,김구원선생
4,스코티시 리더 슈프림,술,기타 주류,ALCOHOL,위스키,스코티시


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1143 entries, 0 to 1142
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   product_name       1143 non-null   object
 1   category_large     1143 non-null   object
 2   category_medium    1143 non-null   object
 3   item_type          1143 non-null   object
 4   grocery_item_name  1143 non-null   object
 5   brand_name         1143 non-null   object
dtypes: object(6)
memory usage: 53.7+ KB


### 브랜드 추출 메서드

In [ ]:
class REF_Brand_Manager:
    def __init__(self):
        # 중복을 허용하지 않는 set 자료형으로 브랜드 사전 초기화
        self.brand_dict = set()
        # 다양한 괄호 패턴 대응 ( [브랜드], (브랜드), 【브랜드】 )
        self.bracket_pattern = r'^[\(\[【](.*?)[\)\]】]'
    
    # 브랜드 추출 메서드
    def extract_brand(self, product_name):
        
        #제품명에서 브랜드를 추출, 브랜드가 제거된 제품명을 반환
        if pd.isna(product_name) or not str(product_name).strip(): # 입력값이 결측치면 예외처리
            return "Unknown", ""
            
        #양쪽 공백 제거, 괄호 패턴 검색
        product_name = str(product_name).strip() 
        match = re.search(self.bracket_pattern, product_name)

        if match:
            # 패턴이 일치 -> 괄호 안의 내용을 추출하여 브랜드로 설정
            brand = match.group(1).strip()
            # 제품명에서 브랜드 영역을 제거
            clean_name = re.sub(self.bracket_pattern, '', product_name).strip()
            
            # 추출된 브랜드가 비어있지 않으면 사전에 추가
            if brand:
                self.brand_dict.add(brand)
        else:
            # 괄호가 없는 경우, 브랜드를 Unknown 으로 표시
            brand = "Unknown"
            clean_name = product_name
        # 브랜드명, 정제명 두가지 리턴
        return brand, clean_name

    
    def build_dictionary_from_df(self, df, column_name='product_name'):
        # product_name 컬럼의 모든 행에 extract_brand 메서드 적용
        results = df[column_name].apply(self.extract_brand)

        # 결과물을 각각 'predict_brand_name'과 'clean_product_name' 컬럼으로 쪼개서 저장
        df['predict_brand_name'] = [r[0] for r in results]
        df['clean_product_name'] = [r[1] for r in results]
        
        print(f"사전 구축 완료: 총 {len(self.brand_dict)}개의 고유 브랜드 발견")
        return df

    def get_brand_list(self):
        #현재까지 구축된 브랜드 리스트를 반환
        return sorted(list(self.brand_dict))

    def is_known_brand(self, brand_name):
        #특정 단어가 이미 등록된 브랜드인지 확인
        return brand_name in self.brand_dict